In [1]:
import os
import sys
import time
from typing import Optional, Sequence, Tuple, List

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv

from openai import OpenAI

# Load environment variables from current directory .env and override existing ones
load_dotenv(".env", override=True)

# load api key from environment
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")
    
client = OpenAI(api_key=api_key)

In [2]:
df = pd.read_json("hf://datasets/HannahRoseKirk/prism-alignment/survey.jsonl", lines=True)

# Use the self-written constitution with respect to how participants want AI Language models to behave

mask_principles = df["system_string"].notna()
self_written_ai_principles = df.loc[mask_principles, "system_string"].tolist()
principle_df_indices = df.index[mask_principles].to_numpy()

In [3]:
def get_embedding(principle):
    num_retries = 0
    max_retries = 5
    while True:
        try:
            response = client.embeddings.create(
                input=principle,
                model="text-embedding-3-small",
            )
            return response.data[0].embedding
        except Exception as e:
            print(f"Error: {e}. Retrying in 5 seconds...")
            num_retries += 1
            if num_retries >= max_retries:
                print("Max retries reached. Exiting.")
                sys.exit(1)
            time.sleep(5)

In [4]:
emb_path = "self_written_ai_principles_embeddings.npy"

if os.path.exists(emb_path):
    print(f"Loading embeddings from {emb_path}")
    X = np.load(emb_path)
    X = np.asarray(X, dtype=np.float32)
else:
    print("No saved embeddings found. Computing embeddings now...")
    embeddings = []
    for p in tqdm(self_written_ai_principles, desc="Computing embeddings"):
        embeddings.append(get_embedding(p))

    X = np.vstack(embeddings)
    np.save(emb_path, X)
    print(f"Saved embeddings to {emb_path}")

print("Embeddings shape:", X.shape)

Loading embeddings from self_written_ai_principles_embeddings.npy
Embeddings shape: (1500, 1536)


In [5]:
def pairwise_distances(points):
    points = np.asarray(points, dtype=np.float32)
    sq_norms = np.sum(points ** 2, axis=1)
    D2 = sq_norms[:, None] + sq_norms[None, :] - 2.0 * (points @ points.T)
    D2 = np.maximum(D2, 0.0)
    return np.sqrt(D2)


def greedy_capture_fair_clustering(points, k):
    """Greedy Capture Algorithhm"""
    points = np.asarray(points, dtype=np.float32)
    n = points.shape[0]
    
    dist = pairwise_distances(points)  

    events = []
    for i in range(n):
        for j in range(n):
            events.append((dist[i, j], i, j))
    events.sort(key=lambda t: t[0])

    quota = int(np.ceil(n / k))
    remaining = set(range(n))
    open_centers = []
    ball_points = [set() for _ in range(n)]

    for d_ij, i, j in events:
        if not remaining:
            break
        if len(open_centers) >= k:
            break
        if i not in remaining:
            continue

        if j in open_centers:
            remaining.remove(i)
        else:
            ball_points[j].add(i)
            if len(ball_points[j]) >= quota:
                open_centers.append(j)
                for p in ball_points[j]:
                    if p in remaining:
                        remaining.remove(p)
                ball_points[j].clear()

    open_centers = np.array(open_centers, dtype=int)

    dist_to_open = dist[:, open_centers]
    labels = np.argmin(dist_to_open, axis=1)

    return open_centers, labels

In [6]:
k = 15

center_idx_gc, labels_gc = greedy_capture_fair_clustering(X, k)

print("Greedy Capture")
print("  num_centers:", len(center_idx_gc))

Greedy Capture
  num_centers: 15


In [7]:
center_df_indices_gc = principle_df_indices[center_idx_gc]

cluster_reps_gc = pd.DataFrame({
    "cluster_id": np.arange(len(center_idx_gc)),
    "df_index": center_df_indices_gc,
    "representative_principle": df.loc[center_df_indices_gc, "system_string"].values,
})

cluster_reps_gc.to_json("cluster_representatives_greedy_capture.json", orient="records", force_ascii=False)

cluster_reps_gc


,cluster_id,df_index,representative_principle
0,0,682,The AI should provide honest and factual infor...
1,1,986,The AI should always behave rationally and nev...
2,2,565,The AI language model should consistently exhi...
3,3,763,The AI model should be kind and helpful. If as...
4,4,576,I believe the AI language model should consist...
5,5,594,The AI should consistently exhibit traits of r...
6,6,697,The AI model should be factual and to the poin...
7,7,605,The AI should come across as friendly yet neut...
8,8,1314,The AI language model should be impartial. It ...
9,9,1435,"The AI should consistently give objective, acc..."


In [9]:
print("15 cluster centers using Greedy Capture")
for princ in cluster_reps_gc["representative_principle"]:
    print("*PRINCIPLE* " + princ + "\n")

15 cluster centers using Greedy Capture
*PRINCIPLE* The AI should provide honest and factual information without bias. AI should also be polite and as concise as possible.

*PRINCIPLE* The AI should always behave rationally and never emotionally or be unreasonable. The communication style should always be clear and as concise as possible, meandering answers are never helpful. The AI should also be able to distinguish the nuances in what might be being asked of it. For example, differences between the literal or the figurative. In regards to what content the AI should not exhibit it would be desirable if the AI was not politically biased to any worldview or ideology. It should be of paramount importance the AI remains objective and truthful.  

*PRINCIPLE* The AI language model should consistently exhibit traits of accuracy, objectivity, and empathy. It must prioritize providing reliable information while avoiding biases or opinions. In professional assistance, maintaining a formal and 